# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [1]:
team_setup = {
    'team_name': 'The Cleaners',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Karen Tran', 'github': 'karen-tran08'},
        {'name': 'Zach Bernas', 'github': 'Zakattak0'},
    ],
    'repo_url': 'https://github.com/karen-tran08/walmart-hurricane-project',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/karen-tran08/walmart-hurricane-project/blob/main/CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Karen',           # nice to have
    'meeting_time': 'Monday, 2PM',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': [
        '1. [The product catalog lists the same product under multiple SKUs, with Strawberry Pop-Tarts alone appearing four times with different costs, suppliers, and package sizes.]',
        '2. [Some transactions have differently formatted store IDs like FL_302, FL302, fl-302, or "FL-302", so the 12 stores show up as 60 distinct values]',
        '3. [The transaction category has 37 different spellings for about 10 categories, like EMERGENCY, emergency, and Emergency Supplies]',
        '4. [Timestamps come in 5 different formats, with many missing, and thousands of prices are stored as text with a dollar sign]',
        '5. [Transactions have multiple fully duplicated rows and over 100 rows have negative quantities]'
    ],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [2]:
import os
import subprocess

# Clone the team repository to the parent directory where the notebook expects it
local_path = '../walmart-hurricane-project'
if not os.path.exists(local_path):
    # Disable interactive terminal prompts so git doesn't hang or fail waiting for login
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    subprocess.run(['git', 'clone', 'https://github.com/karen-tran08/walmart-hurricane-project.git', local_path], env=env)
else:
    print('Repository already cloned locally at', local_path)

Repository already cloned locally at ../walmart-hurricane-project


In [3]:
import subprocess

REPO_PATH = '../walmart-hurricane-project'      # your local clone of the team repo

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

685602f Changed team name to The Cleaners
a4b1c2c Updated notebook owner of the week and meeting time
feb26b9 Add team charter


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [11]:
import re
import pandas as pd

def drop_duplicates(df):
    # Remove exact duplicate rows (same transaction recorded twice)
    return df.drop_duplicates().reset_index(drop=True)

def clean_unit_price(df):
    # Strip '$', commas, and whitespace, then convert unit_price to float
    df = df.copy()
    df["unit_price"] = pd.to_numeric(
        df["unit_price"].astype(str).str.replace(r"[$,\s]", "", regex=True),
        errors="coerce",
    )
    return df

def handle_negative_quantity(df):
    # Drop rows quantities <= 0
    return df[df["quantity"] >= 0].reset_index(drop=True)

def parse_timestamp(df):
    # Parse the five mixed timestamp formats into one datetime column
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", dayfirst=False, errors="coerce")
    return df

def normalize_store_id(value):
    # Turn 'fl_302', 'FL302', ' FL-302' etc. into 'FL-302'
    if pd.isna(value):
        return value
    match = re.fullmatch(r"([A-Z]{2})[-_ ]?(\d+)", str(value).strip().upper())
    return f"{match.group(1)}-{match.group(2)}" if match else str(value).strip().upper()

def clean_store_id(df):
    df = df.copy()
    df["store_id"] = df["store_id"].map(normalize_store_id)
    return df

def clean_stores(stores):
    # Normalize store IDs in the store file and drop the duplicate stores
    stores = clean_store_id(stores)
    return stores.drop_duplicates(subset="store_id").reset_index(drop=True)

def clean_transactions(df):
    print(f"Starting rows:       {len(df):,}\n")

    # Drop Duplicates (Row count change)
    before = len(df)
    df = drop_duplicates(df)
    print(f"  drop_duplicates    -{before - len(df):>4} rows (removed)")

    # Clean Unit Price
    pre_clean = df["unit_price"].astype(str)
    df = clean_unit_price(df)
    modified_prices = (pre_clean != df["unit_price"].astype(str)).sum()
    print(f"  clean_unit_price   ~{modified_prices:>4} rows modified (values formatted)")

    # Handle Negative Quantity (Row count change)
    before = len(df)
    df = handle_negative_quantity(df)
    print(f"  handle_negative_quantity   -{before - len(df):>4} rows (removed)")

    # Parse Timestamp
    pre_clean = df["timestamp"].astype(str)
    df = parse_timestamp(df)
    modified_timestamps = (pre_clean != df["timestamp"].astype(str)).sum()
    print(f"  parse_timestamp    ~{modified_timestamps:>4} rows modified (standardized/parsed)")

    # Clean Store ID
    pre_clean = df["store_id"].astype(str)
    df = clean_store_id(df)
    modified_stores = (pre_clean != df["store_id"].astype(str)).sum()
    print(f"  clean_store_id     ~{modified_stores:>4} rows modified (normalized structure)")

    print(f"\nEnding rows:       {len(df):,}")
    return df

if __name__ == "__main__":
    raw = pd.read_csv("walmart_transactions.csv")
    stores = clean_stores(pd.read_csv("store_locations.csv"))

    txns = clean_transactions(raw)

    # Checks
    unmatched = set(txns["store_id"]) - set(stores["store_id"])
    print(f"\nStore IDs after cleaning: {txns['store_id'].nunique()} (unmatched to store file: {unmatched or 'none'})")
    print(f"Stores in store file after deduplication: {len(stores)}")
    print(f"unit_price dtype: {txns['unit_price'].dtype}, nulls: {txns['unit_price'].isna().sum()}")
    print(f"timestamp dtype: {txns['timestamp'].dtype}, missing: {txns['timestamp'].isna().sum()}")

Starting rows:       25,562

  drop_duplicates    -  30 rows (removed)
  clean_unit_price   ~6424 rows modified (values formatted)
  handle_negative_quantity   - 130 rows (removed)
  parse_timestamp    ~13304 rows modified (standardized/parsed)
  clean_store_id     ~4324 rows modified (normalized structure)

Ending rows:       25,402

Store IDs after cleaning: 12 (unmatched to store file: none)
Stores in store file after deduplication: 12
unit_price dtype: float64, nulls: 0
timestamp dtype: datetime64[ns], missing: 881


### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [5]:
import pandas as pd

decision_log = [
    {
        "Decision #": "001",
        "Action": "Remove exact duplicate rows",
        "Rationale": "Eliminate duplicate transaction entries caused by collection/recording errors.",
        "Rows Affected": "30 removed",
        "Status": "Agreed"
    },
    {
        "Decision #": "002",
        "Action": "Clean and convert unit price to float",
        "Rationale": "Strip currency characters ($, commas) and white space to enable mathematical calculations.",
        "Rows Affected": "6,424 modified",
        "Status": "Agreed"
    },
    {
        "Decision #": "003",
        "Action": "Drop negative quantities",
        "Rationale": "Exclude returns or corrupt transactional lines with quantity <= 0 from active demand calculations.",
        "Rows Affected": "130 removed",
        "Status": "Agreed"
    },
    {
        "Decision #": "004",
        "Action": "Parse multi-format timestamps",
        "Rationale": "Standardize the various formats of timestamps into unified pandas Datetime objects, (881 rows missing in original data).",
        "Rows Affected": "13,304 modified",
        "Status": "Agreed"
    },
    {
        "Decision #": "005",
        "Action": "Normalize store IDs and deduplicate store list",
        "Rationale": "Align inconsistent IDs (e.g., FL302 to FL-302)",
        "Rows Affected": "4,324 modified",
        "Status": "Agreed"
    }
]

df_log = pd.DataFrame(decision_log)
display(df_log)

,Decision #,Action,Rationale,Rows Affected,Status
0,001,Remove exact duplicate rows,Eliminate duplicate transaction entries caused...,30 removed,Agreed
1,002,Clean and convert unit price to float,"Strip currency characters ($, commas) and whit...","6,424 modified",Agreed
2,003,Drop negative quantities,Exclude returns or corrupt transactional lines...,130 removed,Agreed
3,004,Parse multi-format timestamps,Standardize the various formats of timestamps ...,"13,304 modified",Agreed
4,005,Normalize store IDs and deduplicate store list,"Align inconsistent IDs (e.g., FL302 to FL-302)","4,324 modified",Agreed


### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [6]:
# TODO
# not started

### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [7]:
# TODO
# not started

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [8]:
# TODO
# not started

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [9]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': 25402,        # an integer
    'decisions_logged': 5,           # how many entries in your log
    'canonical_products': None,         # distinct products after consolidation
    'weather_api_rows': None,           # rows returned by your API call
    'notebook_runs_clean': False,       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  done - rows_after_cleaning = 25402
  done - decisions_logged = 5
  not yet - canonical_products = None
  not yet - weather_api_rows = None
  not yet - notebook_runs_clean = False

required items complete: True
progress items done: 2 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time.

**b)** What are you doing about it this week? Name who is doing it and by when.


**Biggest risk:** Something that could go wrong is SKU consolidation errors or logic mismatches when mapping in the product catalog. For example, if the string matching rules are too aggressive, it could potentially distort demand calculations.

**Plan, with an owner and a date:**
- **Karen:** Write the consolidation script for Pop-Tarts and evaluate mismatches by Wednesday, Oct 28.
- **Zach:** Manually audit sample consolidated groups and verify mappings against store files by Friday, Oct 30.


In [10]:
with open('../walmart-hurricane-project/CHARTER.md', 'r') as f:
    charter_content = f.read()

print(charter_content[:2000]) # Preview first 2000 characters of the charter

# Team charter: The Cleaners

## Members
- Karen (GitHub: karen-tran08)
- Zach (GitHub: Zakattak0)

## Goals
- **good result:** We will distribute the work evenly and make sure every specification is completed and explainable.
- **what each person wants:** We want to get more practice with APIs and data cleaning, and we want to practice presenting.
- **time limits:** Other coursework will limit the time we can dedicate to this project.
- **what to avoid:** We will avoid waiting until the last minute to complete tasks.

## Roles

| Role | Owner | Backup |
|---|---|---|
| cleaning + decision log | Karen | Zach |
| weather API + join | Zach | Karen |
| analysis Q1-Q5 | Both (split below) | Both |
| charts + presentation | Both | Both |

## Question owners

| Question | Owner | Reviewer |
|---|---|---|
| Q1 demand surge | Karen | Zach |
| Q2 SKU consolidation | Zach | Karen |
| Q3 weather correlation | Zach | Karen |
| Q4 stores and geography | Karen | Zach |
| Q5 signal or artifact | Kare

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | in progress | Karen |
| Q2 SKU consolidation | not started  | Zach |
| Q3 weather correlation | not started | Zach |
| Q4 store and geography | not started | Karen |
| Q5 signal or artifact | not started | Karen |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.